# Czibik, et. al 2021
Measures from Networked corruption risks in European Defense Procurement, 
Czibik, Fazekas, Hernandez Sanchez, Wachs

In [2]:
import pandas as pd
import numpy as np
from pyprojroot import here
import statsmodels.api as sm
import statsmodels.formula.api as smf
#from unidecode import unidecode
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import utils
#from patsy import dmatrices

processed_data = here('data/processed_data')


In [3]:
#load from feather
contracts = pd.read_feather(processed_data / 'mxc_11to22_cri.feather')

In [4]:
contracts

,supplier_name_clean,contract_initial_date,contract_price_mx,purchasing_unit_id,contract_year,CRI,rf_submission_period,rf_decision_period,rf_procedure_type,rf_bl_conformity,rf_buyer_dependence,rf_single_bidder,submission_period_t,decision_period_t,prov_id
0,tcaempresarialsadecv,2011-12-14,146343.00,050GYR005,2011,0.564379,1.0,0.50,1.0,0.25,0.071895,NaN,missing,missing,0
1,instrumentosyaccesoriosautomatizadossadecv,2011-12-15,6250.00,050GYR005,2011,0.550048,1.0,0.50,1.0,0.25,0.000241,NaN,missing,missing,1
2,dacegacorporation,2011-12-22,54400.00,050GYR025,2011,0.657476,1.0,0.25,1.0,1.00,0.037381,NaN,missing,5-13,2
3,lizbethapariciorazo,2011-10-03,88579.20,016000997,2011,0.650060,1.0,0.25,1.0,1.00,0.000300,NaN,missing,5-13,3
4,victormiguelvallejojuarez,2011-09-20,163970.68,821045953,2011,0.580052,1.0,0.25,0.5,1.00,0.150261,NaN,missing,5-13,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2308903,formaseficientessadecv,2022-02-12,364613.67,QDV,2022,0.646566,1.0,0.50,1.0,0.50,0.232832,NaN,missing,missing,2308903
2308904,cosmopapelsadecv,2022-02-14,346016.40,QDV,2022,0.644191,1.0,0.50,1.0,0.50,0.220956,NaN,missing,missing,2308904
2308905,farvisaninsumosinstitucionalessadecv,2022-02-11,1885.35,QDV,2022,0.600241,1.0,0.50,1.0,0.50,0.001204,NaN,missing,missing,2308905
2308906,formaseficientessadecv,2022-02-22,1036815.04,NBD,2022,0.772899,1.0,0.50,1.0,0.50,0.864494,NaN,missing,missing,2308906


## Average CRI per supplier and buyer

In [5]:
#group by year and purchasing unit and get mean of CRI
contracts['buyer_cri'] = contracts.groupby(['contract_year', 'purchasing_unit_id'])['CRI'].transform('mean')


In [6]:
contracts['supplier_cri'] = contracts.groupby(['contract_year', 'supplier_name_clean'])['CRI'].transform('mean')

## Neighborhood corruption risk

In [7]:
bs_cri = contracts.groupby(['contract_year', 'purchasing_unit_id', 'supplier_name_clean'])['CRI'].mean().reset_index()
#rename 
bs_cri = bs_cri.rename(columns={'CRI': 'bs_cri'})
bs_cri.shape

(1059184, 4)

In [8]:
bs_cri.head()

,contract_year,purchasing_unit_id,supplier_name_clean,bs_cri
0,2011,002000999,abastecedoraintegraljmsadecv,0.400004
1,2011,002000999,abastecedorcorporativo,0.266184
2,2011,002000999,abcuniformessadecv,0.510111
3,2011,002000999,abrahamgarciacamacho,0.582163
4,2011,002000999,activecleaningsadecv,0.400127


In [9]:
bs_cri_buyer = bs_cri.groupby(['contract_year', 'purchasing_unit_id']).agg(bs_cri_buyer_sum = ('bs_cri', 'sum'), nsuppliers = ('supplier_name_clean', 'count')).reset_index()


In [10]:
bs_cri_buyer

,contract_year,purchasing_unit_id,bs_cri_buyer_sum,nsuppliers
0,2011,002000999,62.654662,133
1,2011,004000995,84.513333,123
2,2011,004000996,8.000000,13
3,2011,004000997,9.400000,13
4,2011,004000998,29.228200,46
...,...,...,...,...
28310,2022,T0O,2.600000,4
28311,2022,TOM,4.400000,7
28312,2022,TQA,4.400000,7
28313,2022,W3S,5.000000,8


In [11]:
bs_cri_supplier = bs_cri.groupby(['contract_year', 'supplier_name_clean']).agg(bs_cri_supplier_sum = ('bs_cri', 'sum'), nbuyers = ('purchasing_unit_id', 'count')).reset_index()
bs_cri_supplier

,contract_year,supplier_name_clean,bs_cri_supplier_sum,nbuyers
0,2011,0donnellriveranataliemarie,0.632211,1
1,2011,10distribucionesespeciales,0.582068,1
2,2011,130ingenieriayarquitecturasadecv,0.367839,1
3,2011,27micrasinternacional,2.176981,4
4,2011,2gingenieriayarquitcturasadecv,0.474924,2
...,...,...,...,...
629577,2022,zyanyasahiandiazperez,0.551487,1
629578,2022,zyanyaselenecuellarrosales,0.800067,1
629579,2022,zyxasadecv,0.333342,1
629580,2022,zyzlilamargaritagomezarce,0.710276,1


In [12]:
print(bs_cri.shape)
print(bs_cri_buyer.shape)
print(bs_cri_supplier.shape)

(1059184, 4)
(28315, 4)
(629582, 4)


In [13]:
bs_cri = bs_cri.merge(bs_cri_buyer, on=['contract_year', 'purchasing_unit_id'], how='left')

In [14]:
bs_cri = bs_cri.merge(bs_cri_supplier, on=['contract_year', 'supplier_name_clean'], how='left')

In [15]:
bs_cri.shape

(1059184, 8)

In [16]:
bs_cri['bs_cri_no_intersection'] = bs_cri['bs_cri_buyer_sum'] + bs_cri['bs_cri_supplier_sum'] - (2*bs_cri['bs_cri'])

In [17]:
bs_cri['n_neighbors'] = bs_cri['nsuppliers'] + bs_cri['nbuyers'] - 2

In [18]:
bs_cri['neighbourhood_cri'] = bs_cri['bs_cri_no_intersection']  / bs_cri['n_neighbors']

## merge

In [19]:
contracts = contracts.merge(bs_cri[['contract_year', 'purchasing_unit_id', 'supplier_name_clean', 'bs_cri', 'neighbourhood_cri']], on=['contract_year', 'purchasing_unit_id', 'supplier_name_clean'], how='left')

In [20]:
contracts.shape

(2308908, 19)

In [21]:
contracts.columns

Index(['supplier_name_clean', 'contract_initial_date', 'contract_price_mx',
       'purchasing_unit_id', 'contract_year', 'CRI', 'rf_submission_period',
       'rf_decision_period', 'rf_procedure_type', 'rf_bl_conformity',
       'rf_buyer_dependence', 'rf_single_bidder', 'submission_period_t',
       'decision_period_t', 'prov_id', 'buyer_cri', 'supplier_cri', 'bs_cri',
       'neighbourhood_cri'],
      dtype='object')

In [22]:
var2keep = ['supplier_name_clean', 'purchasing_unit_id', 'contract_year',  'buyer_cri', 'supplier_cri', 'bs_cri', 'neighbourhood_cri']

#save to feather
contracts[var2keep].to_feather(processed_data / 'czibik_etal2021.feather')